In [ ]:
!pip install datasets==2.18.0 pytrec_eval pyserini transformers accelerate bitsandbytes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.1/166.1 MB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.5/213.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 1. Ensure Java 21 is installed on the system
!apt-get update -qq
!apt-get install openjdk-21-jdk-headless -qq > /dev/null

import os
import json
from datasets import load_dataset

# 2. Point Python to the new Java installation
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

print("Downloading CIRAL Corpora...")
# Load native Swahili for the LLM mapping
corpus_swahili = load_dataset("CIRAL/ciral-corpus", "swahili", split="train", trust_remote_code=True)

# Load English-translated Swahili for the BM25-DT Index
corpus_translated = load_dataset("CIRAL/ciral-corpus", "swahili", split="train", translated=True, trust_remote_code=True)

# Create a mapping to fetch original Swahili text later during reranking
id_to_swahili = {doc["docid"]: doc["text"] for doc in corpus_swahili}

print("Formatting TRANSLATED corpus for Pyserini...")
os.makedirs("ciral_corpus_json", exist_ok=True)
with open("ciral_corpus_json/collection.jsonl", "w", encoding="utf-8") as f:
    for doc in corpus_translated:
        # We index the ENGLISH translated text
        f.write(json.dumps({"id": doc["docid"], "contents": doc["text"]}) + "\n")

print("Formatting complete. Building BM25-DT Index (This will take a couple of minutes)...")

# 3. Build the Lucene index using Pyserini
!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input ciral_corpus_json \
  --index ciral_swahili_translated_index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 2 --storeRaw

print("\nBM25-DT Index successfully built!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Formatting TRANSLATED corpus for Pyserini...
Formatting complete. Building BM25-DT Index (This will take a couple of minutes)...
[0.051s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.051s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-05-12 11:14:13,908 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - Setting log level to INFO
2026-05-12 11:14:13,913 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) - ============ Loading Index Configuration ============
2026-05-12 11:14:13,913 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) - AbstractIndexer settings:
2026-05-12 11:14:13,915 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + DocumentCollection path: ciral_corpus_json
2026-05-12 11:14:13,916 INFO  [main] i

In [ ]:
import torch
import re
import json
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pyserini.search.lucene import LuceneSearcher
from datasets import load_dataset

# 1. Load Model
print("Loading RankZephyr in 4-bit precision...")
model_id = "castorini/rank_zephyr_7b_v1_full"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

# 2. Point Searcher to the TRANSLATED index
print("Loading BM25 Searcher and Queries...")
searcher = LuceneSearcher('ciral_swahili_translated_index')
queries = load_dataset("CIRAL/ciral", "swahili", split="testA", trust_remote_code=True)

def clean_and_parse_output(raw_output, num_docs):
    cleaned = raw_output.replace(" ", "").replace("▁", "").replace("\n", "")
    raw_indices = [int(x) - 1 for x in re.findall(r'\[(\d+)\]', cleaned)]
    ranked_indices = [idx for idx in raw_indices if idx < num_docs]

    seen = set()
    ranked_indices = [x for x in ranked_indices if not (x in seen or seen.add(x))]

    missing = [i for i in range(num_docs) if i not in ranked_indices]
    ranked_indices.extend(missing)
    return ranked_indices

def build_zephyr_prompt(query, docs):
    prompt = "<|system|>\nYou are RankGPT, an intelligent assistant that can rank passages based on their relevancy to the query.</s>\n<|user|>\n"
    prompt += f"I will provide you with {len(docs)} passages, each indicated by number identifier [].\n"
    prompt += f"Rank the passages based on their relevance to the query: {query}.\n\n"
    for i, doc in enumerate(docs):
        # Because it's English, we can safely bump the text context back up to 800 characters
        prompt += f"[{i+1}] {doc['text'][:800]}\n"
    prompt += f"\nSearch Query: {query}\nRank the {len(docs)} passages above based on their relevance to the search query. "
    prompt += "The passages should be listed in descending order using identifiers. The most relevant passages should be listed first. "
    prompt += "The output format should be [] > [], e.g., [1] > [2]. Only respond with the ranking results, do not say any word or explain.</s>\n<|assistant|>\n["
    return prompt

def rank_window(query, docs_window):
    prompt = build_zephyr_prompt(query, docs_window)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False, temperature=0.0)

    generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    new_order_indices = clean_and_parse_output("[" + generated_text, len(docs_window))

    del inputs, outputs
    torch.cuda.empty_cache()

    return [docs_window[i] for i in new_order_indices]

def sliding_window_rerank(query, initial_ranking, window_size=20, stride=10):
    current_ranking = initial_ranking.copy()
    step = window_size - stride
    for i in range(len(current_ranking), 0, -step):
        start_idx = max(0, i - window_size)
        end_idx = i
        if end_idx - start_idx < 2:
            break
        current_ranking[start_idx:end_idx] = rank_window(query, current_ranking[start_idx:end_idx])
        if start_idx == 0:
            break
    return current_ranking

output_file = "ciral_experiment_english_k20.trec"
print(f"Starting execution loop. Writing results to {output_file}...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Evaluating Queries"):
        qid = q['query_id']
        qtext = q['query']

# EXPERIMENT CHANGE 1: k=20 for massive speedup
        hits = searcher.search(qtext, k=20)

        top_docs = []
        for hit in hits:
            # EXPERIMENT CHANGE 2: Fetch the full doc from the searcher, then parse the raw JSON
            doc = searcher.doc(hit.docid)
            english_text = json.loads(doc.raw())["contents"]
            top_docs.append({"id": hit.docid, "text": english_text})

        # C. Second Stage Reranking
        if len(top_docs) > 0:
            reranked_docs = sliding_window_rerank(qtext, top_docs)
        else:
            reranked_docs = []

        # D. Save in TREC format
        for rank, doc in enumerate(reranked_docs):
            score = 100 - rank
            f.write(f"{qid} Q0 {doc['id']} {rank+1} {score} RankZephyr_English_Exp\n")

print("\nBenchmark Complete! You can now run trec_eval.")

Loading RankZephyr in 4-bit precision...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

...

Generating testA split: 0 examples [00:00, ? examples/s]

Generating testB split: 0 examples [00:00, ? examples/s]

Starting execution loop. Writing results to ciral_experiment_english_k20.trec...


Evaluating Queries:   0%|          | 0/85 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Evaluating Queries: 100%|██████████| 85/85 [34:15<00:00, 24.18s/it]


Benchmark Complete! You can now run trec_eval.


In [ ]:
# 1. Download the official Answer Key (qrels) for the Swahili test set
!wget -q https://huggingface.co/datasets/CIRAL/ciral/resolve/main/ciral-swahili/qrels/qrels.ciral-v1.0-sw-test-a-pools.tsv -O ciral-v1.0-sw-test.qrels

# 2. Run the official TREC evaluation script to calculate nDCG@20
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_experiment_english_k20.trec

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.004s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.2123
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_experiment_english_k20.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
recip_rank            	all	0.4064
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import os
from datasets import load_dataset

# 1. Re-link Java for Pyserini
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

# 2. Re-load the mapping dictionary
print("Loading Swahili mapping...")
corpus_swahili = load_dataset("CIRAL/ciral-corpus", "swahili", split="train", trust_remote_code=True)
id_to_swahili = {doc["docid"]: doc["text"] for doc in corpus_swahili}
print("Ready!")

Loading Swahili mapping...
Ready!


In [ ]:
import torch
import re
import json
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pyserini.search.lucene import LuceneSearcher
from datasets import load_dataset

# 1. Load Model
print("Loading RankZephyr in 4-bit precision...")
model_id = "castorini/rank_zephyr_7b_v1_full"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

# 2. Point Searcher to the TRANSLATED index
print("Loading BM25 Searcher and Queries...")
searcher = LuceneSearcher('ciral_swahili_translated_index')
queries = load_dataset("CIRAL/ciral", "swahili", split="testA", trust_remote_code=True)

# 3. Reload Swahili Mapping (just in case your session cleared it)
corpus_swahili = load_dataset("CIRAL/ciral-corpus", "swahili", split="train", trust_remote_code=True)
id_to_swahili = {doc["docid"]: doc["text"] for doc in corpus_swahili}

def clean_and_parse_output(raw_output, num_docs):
    cleaned = raw_output.replace(" ", "").replace("▁", "").replace("\n", "")
    raw_indices = [int(x) - 1 for x in re.findall(r'\[(\d+)\]', cleaned)]
    ranked_indices = [idx for idx in raw_indices if idx < num_docs]

    seen = set()
    ranked_indices = [x for x in ranked_indices if not (x in seen or seen.add(x))]

    missing = [i for i in range(num_docs) if i not in ranked_indices]
    ranked_indices.extend(missing)
    return ranked_indices

def build_zephyr_prompt(query, docs):
    prompt = "<|system|>\nYou are RankGPT, an intelligent assistant that can rank passages based on their relevancy to the query.</s>\n<|user|>\n"
    prompt += f"I will provide you with {len(docs)} passages, each indicated by number identifier [].\n"
    prompt += f"Rank the passages based on their relevance to the query: {query}.\n\n"
    for i, doc in enumerate(docs):
        # CRITICAL: Truncated to 400 characters to prevent Swahili token explosion
        prompt += f"[{i+1}] {doc['text'][:400]}\n"
    prompt += f"\nSearch Query: {query}\nRank the {len(docs)} passages above based on their relevance to the search query. "
    prompt += "The passages should be listed in descending order using identifiers. The most relevant passages should be listed first. "
    prompt += "The output format should be [] > [], e.g., [1] > [2]. Only respond with the ranking results, do not say any word or explain.</s>\n<|assistant|>\n["
    return prompt

def rank_window(query, docs_window):
    prompt = build_zephyr_prompt(query, docs_window)

    # CRITICAL: Strict max_length=4096 enforcement to prevent GPU OOM
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False, temperature=0.0)

    generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    new_order_indices = clean_and_parse_output("[" + generated_text, len(docs_window))

    del inputs, outputs
    torch.cuda.empty_cache()

    return [docs_window[i] for i in new_order_indices]

def sliding_window_rerank(query, initial_ranking, window_size=20, stride=10):
    # Since k=20, this will effectively only run 1 single pass.
    current_ranking = initial_ranking.copy()
    step = window_size - stride
    for i in range(len(current_ranking), 0, -step):
        start_idx = max(0, i - window_size)
        end_idx = i
        if end_idx - start_idx < 2:
            break
        current_ranking[start_idx:end_idx] = rank_window(query, current_ranking[start_idx:end_idx])
        if start_idx == 0:
            break
    return current_ranking

output_file = "ciral_experiment_crosslingual_k20.trec"
print(f"Starting execution loop. Writing results to {output_file}...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Evaluating Queries"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 hits from the English index
        hits = searcher.search(qtext, k=20)

        top_docs = []
        for hit in hits:
            # EXPERIMENT CHANGE: Map back to the native SWAHILI text
            swahili_text = id_to_swahili.get(hit.docid, "")
            top_docs.append({"id": hit.docid, "text": swahili_text})

        # Second Stage Reranking
        if len(top_docs) > 0:
            reranked_docs = sliding_window_rerank(qtext, top_docs)
        else:
            reranked_docs = []

        # Save in TREC format
        for rank, doc in enumerate(reranked_docs):
            score = 100 - rank
            f.write(f"{qid} Q0 {doc['id']} {rank+1} {score} RankZephyr_CrossLingual\n")

print("\nBenchmark Complete! You can now run trec_eval.")

Loading RankZephyr in 4-bit precision...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading BM25 Searcher and Queries...


Starting execution loop. Writing results to ciral_experiment_crosslingual_k20.trec...


Evaluating Queries:   0%|          | 0/85 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Evaluating Queries: 100%|██████████| 85/85 [34:09<00:00, 24.12s/it]


Benchmark Complete! You can now run trec_eval.


In [ ]:
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_experiment_crosslingual_k20.trec

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.2123
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
# Calculate MRR@100 for the Cross-lingual experiment
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_experiment_crosslingual_k20.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
recip_rank            	all	0.4064
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import torch
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------------------------------------------------
# 1. Load Model & Tokenizer
# ---------------------------------------------------------
# To run AfriMT5, change this to: "castorini/afrimt5-base-ft-msmarco"
model_id = "castorini/mt5-base-ft-msmarco"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda")

# ---------------------------------------------------------
# 2. Pointwise Scoring Function
# ---------------------------------------------------------
def get_pointwise_score(query, doc_text):
    # Formatted exactly as the paper's baselines were trained
    input_text = f"Query: {query} Document: {doc_text}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.no_grad():
        # We only need the model to predict 1 token (the relevance label)
        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            return_dict_in_generate=True,
            output_scores=True
        )
        # Score is the logit of the 'true' token (ID 259)
        # Higher score = more relevant
        score = outputs.scores[0][0, 259]
    return score.item()

# ---------------------------------------------------------
# 3. Execution Loop (Supports both English and Cross-lingual)
# ---------------------------------------------------------
# Set experiment type: "english" or "cross-lingual"
experiment_type = "english"
output_file = f"ciral_mt5_{experiment_type}.trec"

print(f"Running {experiment_type} reranking...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Reranking"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 documents from BM25
        hits = searcher.search(qtext, k=20)

        scored_docs = []
        for hit in hits:
            # PULLING THE CORRECT TEXT:
            if experiment_type == "english":
                # Pull the English Translated version from the index
                doc_text = json.loads(searcher.doc(hit.docid).raw())['contents']
            else:
                # Pull the Native Swahili from the id_to_swahili mapping
                doc_text = id_to_swahili.get(hit.docid, "")

            rel_score = get_pointwise_score(qtext, doc_text)
            scored_docs.append((hit.docid, rel_score))

        # Sort the 20 documents by the new mT5 score
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        for rank, (docid, score) in enumerate(scored_docs):
            f.write(f"{qid} Q0 {docid} {rank+1} {score} mT5_Baseline\n")

print(f"\n{experiment_type.upper()} Benchmark Complete! File saved as {output_file}")

Loading castorini/mt5-base-ft-msmarco...


config.json:   0%|          | 0.00/798 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Running english reranking...


Reranking: 100%|██████████| 85/85 [02:25<00:00,  1.72s/it]


ENGLISH Benchmark Complete! File saved as ciral_mt5_english.trec


In [ ]:
#for English Reranking ndcg and mrr
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_mt5_english.trec

# Calculate MRR@100 for the English mT5 experiment
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_mt5_english.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.1667
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
# 1. Update the experiment type to cross-lingual
experiment_type = "cross-lingual"
output_file = f"ciral_mt5_{experiment_type}.trec"

print(f"Running {experiment_type} reranking...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Reranking"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 documents from BM25
        hits = searcher.search(qtext, k=20)

        scored_docs = []
        for hit in hits:
            # For Cross-lingual: English Query vs Native Swahili Passages
            doc_text = id_to_swahili.get(hit.docid, "")

            rel_score = get_pointwise_score(qtext, doc_text)
            scored_docs.append((hit.docid, rel_score))

        # Sort the 20 documents by the new mT5 score
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        for rank, (docid, score) in enumerate(scored_docs):
            f.write(f"{qid} Q0 {docid} {rank+1} {score} mT5_Baseline\n")

print(f"\nCROSS-LINGUAL Benchmark Complete! File saved as {output_file}")

Running cross-lingual reranking...


Reranking: 100%|██████████| 85/85 [01:26<00:00,  1.02s/it]


CROSS-LINGUAL Benchmark Complete! File saved as ciral_mt5_cross-lingual.trec


In [ ]:
# for cross-lingual Rerandking ndcg and mrr
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_mt5_cross-lingual.trec
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_mt5_cross-lingual.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.1836
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import torch
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load AfriMT5 Model (MS MARCO fine-tuned version)
model_id = "castorini/afrimt5-base-ft-msmarco"

print(f"Loading {model_id} for English Reranking...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda")

# 2. Execution Loop
experiment_type = "english"
output_file = f"ciral_afrimt5_{experiment_type}.trec"

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Reranking (English)"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 hits from BM25
        hits = searcher.search(qtext, k=20)

        scored_docs = []
        for hit in hits:
            # PULLING THE ENGLISH TRANSLATED TEXT
            doc_text = json.loads(searcher.doc(hit.docid).raw())['contents']

            # Pointwise scoring
            input_text = f"Query: {qtext} Document: {doc_text}"
            inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
                # Score is logit of 'true' token (ID 259)
                rel_score = outputs.scores[0][0, 259].item()

            scored_docs.append((hit.docid, rel_score))

        # Sort and write to TREC format
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(scored_docs):
            f.write(f"{qid} Q0 {docid} {rank+1} {score} AfriMT5_English\n")

print(f"\nAFRIMT5 English Reranking Complete! File saved as {output_file}")

Loading castorini/afrimt5-base-ft-msmarco for English Reranking...


config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]


Reranking (English): 100%|██████████| 85/85 [01:37<00:00,  1.15s/it]


AFRIMT5 English Reranking Complete! File saved as ciral_afrimt5_english.trec


In [ ]:
# nDCG@20 for AfriMT5 English
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_afrimt5_english.trec

# MRR for AfriMT5 English
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_afrimt5_english.trec

[0.022s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.022s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.2295
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import torch
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------------------------------------------------
# 1. Load AfriMT5 Model & Tokenizer
# ---------------------------------------------------------
# This checkpoint is AfriMT5-base fine-tuned on MS MARCO
model_id = "castorini/afrimt5-base-ft-msmarco"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda")

# ---------------------------------------------------------
# 2. Execution Loop (Running Cross-Lingual)
# ---------------------------------------------------------
# We focus on Cross-lingual because that's where AfriMT5 shines
experiment_type = "cross-lingual"
output_file = f"ciral_afrimt5_{experiment_type}.trec"

print(f"Running AfriMT5 {experiment_type} reranking...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Reranking"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 documents from BM25
        hits = searcher.search(qtext, k=20)

        scored_docs = []
        for hit in hits:
            # Cross-lingual: English Query vs Native Swahili Passages
            doc_text = id_to_swahili.get(hit.docid, "")

            # Pointwise scoring (logit of token 259)
            input_text = f"Query: {qtext} Document: {doc_text}"
            inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
                rel_score = outputs.scores[0][0, 259].item()

            scored_docs.append((hit.docid, rel_score))

        # Sort and write to file
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(scored_docs):
            f.write(f"{qid} Q0 {docid} {rank+1} {score} AfriMT5_Baseline\n")

print(f"\nAFRIMT5 Benchmark Complete! File saved as {output_file}")

Loading castorini/afrimt5-base-ft-msmarco...


Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running AfriMT5 cross-lingual reranking...


Reranking: 100%|██████████| 85/85 [01:27<00:00,  1.03s/it]


AFRIMT5 Benchmark Complete! File saved as ciral_afrimt5_cross-lingual.trec


In [ ]:
# nDCG@20 for AfriMT5
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_afrimt5_cross-lingual.trec

# MRR for AfriMT5
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_afrimt5_cross-lingual.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.2410
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.51.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


In [ ]:
!pip install -q -U google-generativeai

import google.generativeai as genai
import os
from google.colab import userdata

# Securely fetch your API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
import os
import json
from tqdm import tqdm
from google.genai import Client
from google.colab import userdata

# 1. Initialize the new Client
client = Client(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Pointwise Scoring Function (Updated for new SDK)
def get_gemini_score(query, doc_text):
    prompt = f"""
    You are an expert search evaluator.
    On a scale of 0.0 to 1.0, how relevant is the following document to the search query?
    0.0 means not relevant at all, and 1.0 means perfectly relevant.
    Only output the numerical score.

    Query: {query}
    Document: {doc_text}
    Score:"""

    try:
        # Use the new generate_content method
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=prompt
        )

        # Extract and clean the score
        score_text = response.text.strip()
        # Basic parsing to handle cases where Gemini adds a period or extra space
        score = float(score_text.split()[0])
        return score
    except Exception as e:
        # Fallback for safety
        return 0.0

# 3. Execution Loop (English Reranking)
experiment_type = "english"
output_file = f"ciral_gemini_{experiment_type}.trec"

print(f"Starting Gemini 1.5 Flash {experiment_type} reranking...")

with open(output_file, "w") as f:
    # Testing on queries (Adjust slice as needed)
    for q in tqdm(queries, desc="Evaluating with Gemini"):
        qid = q['query_id']
        qtext = q['query']

        hits = searcher.search(qtext, k=20)
        scored_docs = []

        for hit in hits:
            doc_data = json.loads(searcher.doc(hit.docid).raw())
            doc_text = doc_data['contents']

            score = get_gemini_score(qtext, doc_text)
            scored_docs.append((hit.docid, score))

        # Sort by relevance score
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        for rank, (docid, score) in enumerate(scored_docs):
            f.write(f"{qid} Q0 {docid} {rank+1} {score} Gemini_NewSDK\n")

print(f"\nDone! Evaluation file saved as {output_file}")

Starting Gemini 1.5 Flash english reranking...


Evaluating with Gemini: 100%|██████████| 85/85 [01:57<00:00,  1.39s/it]


Done! Evaluation file saved as ciral_gemini_english.trec


In [ ]:
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_gemini_english.trec
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_gemini_english.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.1870
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import os
import json
from tqdm import tqdm
from google.genai import Client
from google.colab import userdata

# 1. Initialize the Client
client = Client(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Pointwise Scoring Function for Native Language
def get_gemini_crosslingual_score(query, swahili_text):
    # We explicitly tell Gemini that the document is in Swahili
    prompt = f"""
    You are an expert Swahili-English linguistic evaluator.

    Instruction: Evaluate if the Swahili passage contains the specific information requested in the English query.
    Note: Swahili uses noun classes; pay close attention to the subject-verb agreement to ensure the context is correct.

    English Query: {query}
    Swahili passage: {swahili_text}

    Return only a probability score (0.0 to 1.0) of relevance.
    Score:"""

    try:
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=prompt
        )
        score_text = response.text.strip()
        # Parse the first numerical value found
        score = float(score_text.split()[0])
        return score
    except Exception:
        return 0.0

# 3. Execution Loop (Cross-lingual)
experiment_type = "cross-lingual"
output_file = f"ciral_gemini_{experiment_type}.trec"

print(f"Starting Gemini 1.5 Flash {experiment_type} reranking...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Evaluating (Cross-lingual)"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 hits from BM25
        hits = searcher.search(qtext, k=20)
        scored_docs = []

        for hit in hits:
            # PULLING THE NATIVE SWAHILI TEXT
            doc_text = id_to_swahili.get(hit.docid, "")

            score = get_gemini_crosslingual_score(qtext, doc_text)
            scored_docs.append((hit.docid, score))

        # Sort by relevance score
        scored_docs.sort(key=lambda x: x[1], reverse=True)

        for rank, (docid, score) in enumerate(scored_docs):
            f.write(f"{qid} Q0 {docid} {rank+1} {score} Gemini_Crosslingual\n")

print(f"\nDone! Cross-lingual file saved as {output_file}")

Starting Gemini 1.5 Flash cross-lingual reranking...


Evaluating (Cross-lingual): 100%|██████████| 85/85 [02:03<00:00,  1.46s/it]


Done! Cross-lingual file saved as ciral_gemini_cross-lingual.trec


In [ ]:
# nDCG@20 for Gemini Cross-lingual
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_gemini_cross-lingual.trec

# MRR for Gemini Cross-lingual
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_gemini_cross-lingual.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.1870
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import re

# 1. New Listwise Ranking Function
def get_gemini_listwise_rank(query, doc_texts):
    """
    Reranks a list of documents by comparing them to each other.
    """
    # Format the documents for the prompt
    doc_entries = "\n".join([f"[{i+1}] {text}" for i, text in enumerate(doc_texts)])

    prompt = f"""
    You are an expert multilingual search evaluator.
    Rank the following 5 Swahili passages based on their relevance to the English query.
    Note: Focus on linguistic nuances and subject-verb agreement in Swahili.

    English Query: {query}

    Passages:
    {doc_entries}

    Output the ranking as a list of indices separated by '>', from most relevant to least relevant.
    Example Output: [3] > [1] > [4] > [2] > [5]
    Ranking:"""

    try:
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=prompt
        )
        # Extract indices like [3], [1], etc.
        rank_string = response.text.strip()
        indices = [int(i) - 1 for i in re.findall(r'\[(\d+)\]', rank_string)]
        return indices
    except Exception:
        return list(range(len(doc_texts))) # Return original order if it fails

# 2. Updated Execution Loop (Sliding Window for Top 20)
experiment_type = "listwise_cross-lingual"
output_file = f"ciral_gemini_{experiment_type}.trec"

print(f"Starting Gemini 1.5 Flash Listwise reranking...")

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="Listwise Reranking"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 hits from BM25
        hits = searcher.search(qtext, k=20)
        current_rank = [hit.docid for hit in hits]

        # SLIDING WINDOW: We process in windows of 5 to break ties
        # We start from the back to push the best results to the front
        for i in range(len(current_rank) - 5, -1, -5):
            window_ids = current_rank[i:i+5]
            window_texts = [id_to_swahili.get(docid, "") for docid in window_ids]

            # Get the new relative order for these 5
            new_order_indices = get_gemini_listwise_rank(qtext, window_texts)

            # Reorder the current_rank list based on Gemini's preference
            reordered_window = [window_ids[idx] for idx in new_order_indices if idx < len(window_ids)]

            # Update the main list
            current_rank[i:i+len(reordered_window)] = reordered_window

        # Write to TREC
        for rank, docid in enumerate(current_rank):
            # We use a descending score (20, 19, 18...) to ensure proper order
            f.write(f"{qid} Q0 {docid} {rank+1} {20-rank} Gemini_Listwise\n")

print(f"\nDone! Listwise file saved as {output_file}")

Starting Gemini 1.5 Flash Listwise reranking...


Listwise Reranking: 100%|██████████| 85/85 [00:29<00:00,  2.89it/s]


Done! Listwise file saved as ciral_gemini_listwise_cross-lingual.trec


In [ ]:
# 1. Calculate nDCG@20 for Gemini Listwise
print("--- nDCG@20 Results ---")
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_gemini_listwise_cross-lingual.trec

# 2. Calculate MRR for Gemini Listwise
print("\n--- MRR Results ---")
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_gemini_listwise_cross-lingual.trec

--- nDCG@20 Results ---
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.2123
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/li

In [ ]:
import torch
from tqdm import tqdm

# 1. Load AfriMT5 (using the same checkpoint)
model_id = "castorini/afrimt5-base-ft-msmarco"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda")

# 2. Pairwise Comparison Function
def compare_docs_afri(query, doc1, doc2):
    """
    Returns the logit difference between two documents.
    If positive, doc1 is better. If negative, doc2 is better.
    """
    def get_score(q, d):
        input_text = f"Query: {q} Document: {d}"
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
            return outputs.scores[0][0, 259].item()

    return get_score(query, doc1) - get_score(query, doc2)

# 3. Execution Loop: Bubble Sort Listwise Reranking
experiment_type = "listwise_afrimt5"
output_file = f"ciral_{experiment_type}.trec"

with open(output_file, "w") as f:
    for q in tqdm(queries, desc="AfriMT5 Listwise"):
        qid = q['query_id']
        qtext = q['query']

        # Get top 20 hits
        hits = searcher.search(qtext, k=20)
        current_rank = [hit.docid for hit in hits]

        # We perform a partial 'Bubble Sort' on the Top 10 to ensure
        # the absolute best documents rise to the very top.
        # (Doing all 20 would take too long, Top 10 is the strategy for T4 GPUs)
        n = 10
        for i in range(n):
            for j in range(0, n - i - 1):
                doc_a = id_to_swahili.get(current_rank[j], "")
                doc_b = id_to_swahili.get(current_rank[j+1], "")

                # If doc_b is better than doc_a, swap them
                if compare_docs_afri(qtext, doc_a, doc_b) < 0:
                    current_rank[j], current_rank[j+1] = current_rank[j+1], current_rank[j]

        # Write result
        for rank, docid in enumerate(current_rank):
            f.write(f"{qid} Q0 {docid} {rank+1} {20-rank} AfriMT5_Listwise\n")

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
AfriMT5 Listwise: 100%|██████████| 85/85 [06:29<00:00,  4.59s/it]


In [ ]:
# nDCG@20 for AfriMT5 Listwise
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.20 ciral-v1.0-sw-test.qrels ciral_listwise_afrimt5.trec

# MRR for AfriMT5 Listwise
!python -m pyserini.eval.trec_eval -c -M 100 -m recip_rank ciral-v1.0-sw-test.qrels ciral_listwise_afrimt5.trec

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_20           	all	0.2376
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-packag

In [ ]:
import pandas as pd
from IPython.display import display

# Manually entering the results we captured across your successful experiments
results_data = {
    "Model": [
        "RankZephyr", "mT5", "AfriMT5", "Gemini 1.5 Flash", "Gemini 1.5 Flash (Listwise)",
        "AfriMT5 (Listwise)", "mT5", "AfriMT5", "Gemini 1.5 Flash", "Gemini 1.5 Flash (Listwise)"
    ],
    "Experiment": [
        "English Reranking", "English Reranking", "English Reranking", "English Reranking", "English Reranking",
        "Cross-lingual", "Cross-lingual", "Cross-lingual", "Cross-lingual", "Cross-lingual"
    ],
    "Strategy": [
        "Listwise", "Pointwise", "Pointwise", "Pointwise", "Listwise",
        "Pairwise", "Pointwise", "Pointwise", "Pointwise", "Listwise"
    ],
    "nDCG@20": [
        0.2123, 0.1667, 0.2295, 0.1870, 0.2123,
        0.2376, 0.1836, 0.2410, 0.1870, 0.2123
    ],
    "MRR@20": [
        0.4064, 0.2247, 0.4584, 0.3179, 0.4064,
        0.5767, 0.2981, 0.5516, 0.3179, 0.4064
    ]
}

# Create DataFrame
df = pd.DataFrame(results_data)

# Sort by Experiment and then nDCG to see top performers
df = df.sort_values(by=["Experiment", "nDCG@20"], ascending=[True, False])

print("--- CIRAL Swahili Reranking Leaderboard (k=20) ---")
display(df.style.highlight_max(subset=["nDCG@20", "MRR@20"], color='lightgreen', axis=0))

# Quick summary for your report
top_model = df.iloc[0]["Model"]
top_score = df.iloc[0]["nDCG@20"]

print(f"\nBest Overall Performance: {top_model} in {df.iloc[0]['Experiment']} ({top_score} nDCG)")

--- CIRAL Swahili Reranking Leaderboard (k=20) ---


,Model,Experiment,Strategy,nDCG@20,MRR@20
7,AfriMT5,Cross-lingual,Pointwise,0.241000,0.551600
5,AfriMT5 (Listwise),Cross-lingual,Pairwise,0.237600,0.576700
9,Gemini 1.5 Flash (Listwise),Cross-lingual,Listwise,0.212300,0.406400
8,Gemini 1.5 Flash,Cross-lingual,Pointwise,0.187000,0.317900
6,mT5,Cross-lingual,Pointwise,0.183600,0.298100
2,AfriMT5,English Reranking,Pointwise,0.229500,0.458400
0,RankZephyr,English Reranking,Listwise,0.212300,0.406400
4,Gemini 1.5 Flash (Listwise),English Reranking,Listwise,0.212300,0.406400
3,Gemini 1.5 Flash,English Reranking,Pointwise,0.187000,0.317900
1,mT5,English Reranking,Pointwise,0.166700,0.224700



Best Overall Performance: AfriMT5 in Cross-lingual (0.241 nDCG)
